# 4 · Lead Qualifier Agent
## Rubric-Driven Scoring and Grounded Reasoning
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/4-lead-qualifier/lead_qualifier_workbook.ipynb)

Sales teams waste enormous time chasing leads that will never buy. The typical solution — a CRM score based on web traffic and job title — doesn't explain *why* a lead is good or bad. A rep has to guess.

This example builds an agent that scores leads against an explicit **Ideal Customer Profile (ICP)** and returns not just a score, but the exact criteria met and missed. Every decision is explainable. The rubric lives in the system prompt — changing business rules means editing text, not code.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The business problem: unqualified pipeline |
| 2 | What is an ICP and why does it belong in the prompt? |
| 3 | Designing a schema that forces grounded reasoning |
| 4 | Building the qualifier with a system-prompt rubric |
| 5 | Running and interpreting results |
| ★ | Exercise: customize the ICP, test new leads |

---

### Prerequisites
- Python 3.10+
- `OPENAI_API_KEY` in `.env` or Colab Secrets

### Key References
> Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS 2022. https://arxiv.org/abs/2201.11903  
> LangChain LCEL (pipe operator): https://python.langchain.com/docs/concepts/lcel/

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded API key from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded API key from .env file.")

key = os.environ.get("OPENAI_API_KEY", "")
if key and key.startswith("sk-"):
    print("✓ OPENAI_API_KEY is set and looks valid.")
else:
    print("✗ OPENAI_API_KEY missing or invalid.")
    print("  Colab: Secrets panel (key icon) → Add secret 'OPENAI_API_KEY'")
    print("  Local: create a .env file with OPENAI_API_KEY=sk-...")

---

## Part 1 — The Business Problem

---

A B2B SaaS company's sales team reviews 50 inbound leads per week. Without qualification, every lead gets the same treatment: a 30-minute discovery call. That's 25 hours per week on calls — many of which are clearly wrong-fit before they start.

A qualification agent pre-scores each lead before a human sees it:

```
lead description  →  ICP qualification agent  →  LeadScore
                                                    ├── score: 8/10
                                                    ├── tier: "hot"
                                                    ├── criteria_met: ["FinTech", "VP Ops", "budget signal"]
                                                    ├── criteria_missed: []
                                                    └── reasoning: "Strong ICP fit: FinTech industry, 
                                                                    decision maker identified, existing 
                                                                    software spend >$5k/month"
```

The rep now spends 30 minutes only on the `hot` leads. `warm` leads get a short email. `cold` leads get added to a nurture sequence.

---

## Part 2 — The ICP: Why It Belongs in the Prompt

---

An Ideal Customer Profile (ICP) is a description of the customers most likely to buy, succeed, and renew. A typical ICP has 4–6 criteria:

```
Industry:      SaaS, FinTech, or E-commerce
Company size:  50–500 employees  
Pain point:    manual workflows, data silos, or compliance burden
Buyer role:    VP Operations, CFO, or CTO
Budget signal: existing software spend > $5k/month
```

**Why put it in the system prompt?**

| Option | How to change the ICP |
|--------|----------------------|
| ICP hardcoded in schema | Edit Python, redeploy |
| ICP hardcoded in a prompt string | Edit Python, redeploy |
| **ICP in system prompt variable** | **Edit a text string, no redeploy** |

The rubric is business logic, not code logic. Sales ops can update the ICP directly — no engineer needed.

---

## Part 3 — A Schema That Forces Grounded Reasoning

---

The key design choice in this example: `criteria_met` and `criteria_missed` are **required fields**.

```python
criteria_met: List[str]    # which ICP criteria this lead satisfies
criteria_missed: List[str] # which ICP criteria this lead doesn't satisfy
```

The model *cannot* return a score without also listing the reasons. This eliminates the black-box score problem — every `8/10` is accompanied by a traceable list of criteria.

This is the **grounded reasoning** pattern: constrain not just the *output shape*, but the *reasoning the model must show*.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field
import json


class LeadScore(BaseModel):
    company: str = Field(description="Name of the company being evaluated")
    score: int = Field(
        ge=1, le=10,
        description="ICP fit score from 1 (no fit) to 10 (perfect fit)"
    )
    tier: Literal["hot", "warm", "cold"] = Field(
        description="hot = score 8-10, warm = score 5-7, cold = score 1-4"
    )
    criteria_met: List[str] = Field(
        description="List of ICP criteria this lead clearly satisfies. "
                    "Name each criterion exactly as it appears in the ICP rubric."
    )
    criteria_missed: List[str] = Field(
        description="List of ICP criteria this lead clearly does NOT satisfy. "
                    "Name each criterion exactly as it appears in the ICP rubric."
    )
    recommended_action: str = Field(
        description="One concrete next step for the sales rep"
    )
    reasoning: str = Field(
        description="2-3 sentences explaining the score, citing specific criteria met and missed"
    )


print("LeadScore JSON Schema:")
print(json.dumps(LeadScore.model_json_schema(), indent=2))

---

## Part 4 — Building the Qualifier

---

The ICP rubric goes into a `SystemMessage`. The lead description goes into a `HumanMessage`. The model must use the rubric to score the lead — it can't ignore criteria it doesn't know about.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

ICP_RUBRIC = """You are a sales qualification assistant. Score inbound leads against this ICP:

IDEAL CUSTOMER PROFILE (ICP):
  1. Industry:      SaaS, FinTech, or E-commerce
  2. Company size:  50–500 employees
  3. Pain point:    manual workflows, data silos, or compliance burden
  4. Buyer role:    VP Operations, CFO, or CTO
  5. Budget signal: existing software spend > $5k/month

SCORING GUIDE:
  8–10 → hot   (3+ criteria met, strong pain + budget signal present)
  5–7  → warm  (2 criteria met, or strong pain but budget unclear)
  1–4  → cold  (fewer than 2 criteria met)

IMPORTANT:
- Populate criteria_met and criteria_missed using the exact criterion labels above (e.g. "Industry", "Buyer role")
- reasoning must cite specific criteria by name
- Never invent data not present in the lead description"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qualifier = SystemMessage(ICP_RUBRIC) | llm.with_structured_output(LeadScore)

print("Qualifier pipeline: ICP system prompt | LLM + LeadScore schema")

---

## Part 5 — Running the Qualifier

---

In [ ]:
LEADS = [
    ("Meridian Payments", """
Company:  Meridian Payments
Industry: FinTech
Size:     120 employees
Contact:  Sarah Chen, VP of Operations

Notes: Sarah mentioned their team reconciles invoices manually across three
spreadsheets — costing ~15 hours/week. They pay $8k/month across various SaaS tools
and want to consolidate before Q3. Budget: $2k–4k/month for the right solution."""),

    ("Corner Bakery Co", """
Company:  Corner Bakery Co
Industry: Food & Beverage retail
Size:     12 employees
Contact:  Mark, Owner

Notes: Looking for a simple way to track daily sales. Currently uses a paper
ledger. No software subscriptions. Budget is tight — hoping to keep it under $50/month."""),

    ("ShopFlow Commerce", """
Company:  ShopFlow Commerce
Industry: E-commerce enablement (SaaS)
Size:     280 employees
Contact:  David Park, CFO

Notes: Compliance burden is growing fast as they expand to EU markets — GDPR and
VAT reporting is manual right now. Current software spend is $22k/month.
David has budget authority and wants a solution in 60 days."""),
]

for name, lead in LEADS:
    result = qualifier.invoke(HumanMessage(lead))
    print(f"{'='*60}")
    print(f"Company:  {result.company}")
    print(f"Score:    {result.score}/10  →  {result.tier.upper()}")
    print(f"Met:      {', '.join(result.criteria_met) or 'none'}")
    print(f"Missed:   {', '.join(result.criteria_missed) or 'none'}")
    print(f"Action:   {result.recommended_action}")
    print(f"Reason:   {result.reasoning}")
    print()

---

## Exercise — Customize the ICP and Test Edge Cases

**Part A:** Update the ICP rubric to match a *different* business. For example:
- A cybersecurity company targeting healthcare (50–200 beds, CISO or CTO, HIPAA pain)
- A logistics SaaS targeting mid-market manufacturers (100–1000 employees, Ops Director)

**Part B:** Write a "deliberately ambiguous" lead — one where 2–3 criteria could go either way. Run it and check whether `reasoning` explains the ambiguity.

**Questions to consider:**
1. Does the model correctly cite criterion labels from your updated rubric?
2. Does a `warm` lead clearly show which criteria are uncertain vs. confirmed missing?

In [ ]:
# ── Exercise Starter ────────────────────────────────────────────────────────

MY_ICP = """You are a sales qualification assistant. Score inbound leads against this ICP:

IDEAL CUSTOMER PROFILE (ICP):
  1. Industry:     TODO: your target industry
  2. Company size: TODO: your target size range
  3. Pain point:   TODO: the problem you solve
  4. Buyer role:   TODO: who signs the deal
  5. Budget signal: TODO: spending indicator

SCORING GUIDE:
  8–10 → hot   (3+ criteria met)
  5–7  → warm  (2 criteria met)
  1–4  → cold  (fewer than 2 criteria met)

Always cite criterion labels by name in criteria_met and criteria_missed."""

# TODO: bind MY_ICP to a qualifier pipeline
# my_qualifier = SystemMessage(MY_ICP) | llm.with_structured_output(LeadScore)

# TODO: write a test lead and run it
# MY_LEAD = "..."
# result = my_qualifier.invoke(HumanMessage(MY_LEAD))

In [ ]:
# ── Answer Key ────────────────────────────────────────────────────────────────
# ICP: cybersecurity SaaS targeting healthcare (50-200 bed hospitals, CISO/CTO)

CYBER_ICP = SystemMessage("""You are a sales qualification assistant for a cybersecurity SaaS
company. Score inbound leads against this Ideal Customer Profile (ICP):

IDEAL CUSTOMER PROFILE (ICP):
  1. Industry:      Healthcare (hospitals, clinics, health systems)
  2. Company size:  50-200 beds or 200-2000 employees
  3. Pain point:    HIPAA compliance gaps, ransomware risk, or legacy security tools
  4. Buyer role:    CISO, CTO, or VP of IT Security
  5. Budget signal: Recent breach, audit finding, or board-level security initiative

SCORING GUIDE:
  8-10 -> hot   (4-5 criteria met, clear pain + budget signal)
  5-7  -> warm  (2-3 criteria, some uncertainty)
  1-4  -> cold  (fewer than 2 criteria met)

Always cite criterion labels by name in criteria_met and criteria_missed.""")

cyber_qualifier = CYBER_ICP | llm.with_structured_output(LeadScore)

# Ambiguous lead: industry match, size uncertain, pain implied but not stated
AMBIGUOUS_LEAD = HumanMessage("""
Company:  Lakeside Regional Medical Center
Industry: Healthcare (regional hospital network)
Size:     Not disclosed -- roughly 3 facilities
Contact:  VP of Operations (not IT or security)
Notes:    Recently migrated EHR to cloud. Board is asking about cyber risk at
          next quarterly meeting. No budget mentioned.
""")

result = cyber_qualifier.invoke(AMBIGUOUS_LEAD)
print("Lead:        Lakeside Regional Medical Center")
print(f"Score:       {result.score}/10  ({result.tier})")
print(f"Met:         {', '.join(result.criteria_met)}")
print(f"Missed:      {', '.join(result.criteria_missed)}")
print(f"Reasoning:   {result.reasoning}")
print(f"Next action: {result.recommended_next_action}")


---

## What's Next?

- **Example 5 — Support Ticket Router** ([`../5-support-ticket-router/`](../5-support-ticket-router/)): add conditional routing after classification — the agent picks the right team and drafts a first response, all in one pass.
- **Example 8 — Multi-Agent Research** ([`../8-multi-agent-research/`](../8-multi-agent-research/)): a supervisor agent hands off to specialized sub-agents. The ICP qualifier pattern becomes one node in a larger pipeline.

### Further Reading
- Wei et al. (2022). *Chain-of-Thought Prompting.* https://arxiv.org/abs/2201.11903
- LCEL (pipe operator): https://python.langchain.com/docs/concepts/lcel/
- LangGraph conditional edges: https://langchain-ai.github.io/langgraph/how-tos/branching/

---
*Workshop complete. Next: Example 5 — Support Ticket Router.*